In [1]:
import torch.nn as nn
from transformers import AutoModelForCausalLM

In [2]:
model_name = "HuggingFaceTB/SmolLM-135M"
model = AutoModelForCausalLM.from_pretrained(model_name)

In [3]:
# ONly tarket linear layers (Because LoRA does that)
target_modules = set()
for name, module in model.named_modules():
    if isinstance(module, nn.Linear):
        target_modules.add(name.split(".")[-1])

target_modules

{'down_proj',
 'gate_proj',
 'k_proj',
 'lm_head',
 'o_proj',
 'q_proj',
 'up_proj',
 'v_proj'}

In [4]:
# What does the above mean?
# It means that these are the linear layers of the model 
# which can be fine tuned.

# by default LoRA only adapts the value and query layers.

# Get average Title length

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer
from huggingface_hub import login

df = pd.read_csv("../1_preprocessing/cleaned/train_data.csv")

login("redacted")

tokenizer = AutoTokenizer.from_pretrained(
    "HuggingFaceTB/SmolLM-135M"
)

titles = df["title"].fillna("").tolist()

batch_size = 1000
lengths = []

for i in range(0, len(titles), batch_size):
    batch = titles[i:i+batch_size]

    enc = tokenizer(
        batch,
        add_special_tokens=False,
        truncation=False
    )

    lengths.extend(len(ids) for ids in enc["input_ids"])

lengths = np.array(lengths)

# Cap everything above 40 at 40
capped_lengths = np.clip(lengths, None, 40)

# Fit normal distribution on capped data
mu = capped_lengths.mean()
sigma = capped_lengths.std()

print("Average (capped) title token length:", mu)
print("Std dev:", sigma)

# Generate plottable normal distribution
x = np.linspace(0, 40, 1000)

pdf = (
    1 / (sigma * np.sqrt(2*np.pi))
    * np.exp(-(x - mu)**2 / (2 * sigma**2))
)

# Plot histogram + normal fit
# Bin edges make 40 the "40 and above" bucket
bins = np.arange(0, 42) - 0.5

plt.style.use('ggplot')

plt.hist(
    capped_lengths,
    bins=bins,
    density=True,
    alpha=0.6,
    label="Observed (40 = 40+)"
)

plt.plot(x, pdf, linewidth=2, label="Normal fit")

plt.xlabel("Title Token Length")
plt.ylabel("Density")
plt.title("Distribution of Title Token Lengths")
plt.legend()
plt.show()

HTTPError: Invalid user token.